# Traffic Speed Analysis - Duval County

Calls the FastAPI microservice in this repo and visualizes the results with MapboxGL.

**Requires:**
- The API running (`docker compose up` from the repo root)
- A free Mapbox access token (https://mapbox.com), set via the `MAPBOX_TOKEN` environment variable

In [ ]:
%pip install -q requests pandas geopandas mapboxgl shapely "ipython<9"

In [ ]:
import os

import geopandas as gpd
import pandas as pd
import requests
from mapboxgl.utils import create_color_stops
from mapboxgl.viz import ChoroplethViz

MAPBOX_TOKEN = os.environ.get("MAPBOX_TOKEN", "")
BASE_URL = os.environ.get("API_BASE_URL", "http://localhost:8000/api/v1")

if not MAPBOX_TOKEN:
    print("Set the MAPBOX_TOKEN environment variable")

## `GET /aggregates/`

In [ ]:
params = {"day": "Monday", "period": "AM Peak"}
response = requests.get(f"{BASE_URL}/aggregates/", params=params)
response.raise_for_status()
geojson_data = response.json()

In [ ]:
features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "link_id": f["link_id"],
            "road_name": f["road_name"],
            "average_speed": f["average_speed"],
            "length": f["length"],
        },
    }
    for f in geojson_data
]

viz = ChoroplethViz(
    {"type": "FeatureCollection", "features": features},
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=create_color_stops([10, 20, 30, 40, 50], colors="Reds"),
    center=(-81.6557, 30.3322),
    zoom=11,
    line_width=1.5,
    opacity=0.8,
)
viz.show()

## Tabular Summary

In [ ]:
df = pd.DataFrame([
    {
        "link_id": f["link_id"],
        "avg_speed": f["average_speed"],
        "road_name": f["road_name"],
        "length": f["length"]
    } for f in geojson_data
])
df.sort_values("avg_speed").head(10)